
# Mini‑sistema HR (Pandas + OOP) — Guida passo‑passo 📘

Quaderno di supporto per l’esercizio **Gestione Dipendenti e Stipendi**. Qui trovi spiegazioni, mini‑esempi e checklist per implementare i **metodi** e le **estensioni** richieste nei file del progetto (`hr_domain.py`, `hr_analysis.py`).

> Obiettivo: completare i TODO senza rompere le API usate dai test e mantenere l’output compatibile.



## Come usare questo notebook (sul tuo progetto locale)
1. Crea e attiva un virtual env, poi installa le dipendenze:
   ```bash
   python -m venv .venv
   source .venv/bin/activate  # Windows: .venv\Scripts\Activate.ps1
   python -m pip install --upgrade pip
   pip install -r requirements.txt  # contiene almeno pandas
   ```
2. Lavora nei file **del progetto** con il tuo editor:
   - `hr_domain.py` → logica OOP (classi, validazioni, metodi)
   - `hr_analysis.py` → pipeline pandas e report
3. Esegui i test da terminale:
   ```bash
   python esercizio_09_hr_app.py
   # quando richiesto, premi Invio
   ```
4. Usa questo notebook solo come **guida**: gli esempi sono dimostrativi e non vanno copiati 1:1.



## Roadmap delle implementazioni
**Dominio OOP (`hr_domain.py`):**
- `Worker.validate()` → aggiungi controlli di **dominio** (range, valori permessi).
- `Employee` → property opzionale `annual_bonus_value` (valore in € del bonus).
- `Manager` → conferma/sperimenta extra **+5%** (già presente) o rendilo variabile per `level`.
- Nuova sottoclasse suggerita: `SeniorManager` con extra **+10%** (override di `total_compensation()`).
- Cura i metodi speciali: `__str__` per stampa leggibile (senza rompere `__repr__`).

**Pandas & Report (`hr_analysis.py`):**
- `load_employees_csv()` → validazioni schema + **valori ammessi**, tipizzazioni, univocità `employee_id`.
- `compute_total_compensation()` → estendi con **seniority** (da `hire_date`), limiti/clip del bonus, regole per `contract_type`.
- `department_stats()` → aggiungi **std**, **IQR**, **rank** e **ordinamento** desc su media.
- `generate_reports()` → oltre a CSV+MD, esporta **JSON** e un grafico a barre (matplotlib).



# Parte A — Dominio OOP (`hr_domain.py`)

### 1) Validazioni in `Worker.validate()`
Punta a **fallire presto** con messaggi chiari. Esempi di regole tipiche:
- `role.level >= 1`
- `0 <= bonus_percent <= 100`
- `0 < workload_percent <= 1`
- `department.cost_center` non vuoto
- `currency in {"EUR","USD","RON"}`

Suggerimento: alza le eccezioni di **dominio** già presenti (`InvalidSalaryError`, `HRDataError`, …).


In [ ]:

# ESEMPIO DIMOSTRATIVO (ridotto): pattern per validazione
class HRDataError(Exception): ...
class InvalidSalaryError(HRDataError): ...

class Role:
    def __init__(self, name, level):
        self.name = name
        self.level = level

class Department:
    def __init__(self, name, cost_center):
        self.name = name
        self.cost_center = cost_center

class WorkerDemo:
    def __init__(self, base_salary, bonus_percent, workload_percent, currency, role, department):
        self.base_salary = base_salary
        self.bonus_percent = bonus_percent
        self.workload_percent = workload_percent
        self.currency = currency
        self.role = role
        self.department = department
        self.validate()
    def validate(self):
        if self.base_salary <= 0:
            raise InvalidSalaryError("Base salary deve essere > 0")
        if self.role.level < 1:
            raise HRDataError("Il level del ruolo deve essere >= 1")
        if not (0 <= self.bonus_percent <= 100):
            raise HRDataError("bonus_percent deve essere tra 0 e 100")
        if not (0 < self.workload_percent <= 1):
            raise HRDataError("workload_percent deve essere (0,1]")
        if self.currency not in {"EUR","USD","RON"}:
            raise HRDataError("Valuta non supportata")
        if not self.department.cost_center:
            raise HRDataError("cost_center mancante")
            
# Prova: questo solleva perché bonus non valido
try:
    WorkerDemo(50000, 150, 1.0, "EUR", Role("Dev", 2), Department("Eng","CC100"))
except Exception as e:
    print("Errore atteso:", e)



### 2) `Employee.annual_bonus_value` (property)
Calcola il valore in **€** del bonus annuale usando `base_salary`, `workload_percent` e `bonus_percent`.


In [ ]:

# ESEMPIO DIMOSTRATIVO: @property per valore del bonus
class EmployeeDemo(WorkerDemo):
    @property
    def annual_bonus_value(self):
        base = self.base_salary * self.workload_percent
        return base * (self.bonus_percent / 100.0)

e = EmployeeDemo(60000, 10, 0.8, "EUR", Role("Dev",2), Department("Eng","CC100"))
print("Bonus annuo (€):", round(e.annual_bonus_value, 2))



### 3) Override di `total_compensation()` e sottoclassi
- `Employee.total_compensation()` → **base × (1 + bonus%)**
- `Manager.total_compensation()` → `super(...)*1.05` (extra 5% già previsto)
- `SeniorManager` (suggerita) → `super(...)*1.10`

> Mantieni le **API** uguali, così i test continuano a funzionare.


In [ ]:

# ESEMPIO DIMOSTRATIVO: overriding del calcolo
class EmployeeCalc:
    def __init__(self, base_salary, bonus_percent, workload_percent=1.0):
        self.base_salary = base_salary
        self.bonus_percent = bonus_percent
        self.workload_percent = workload_percent
    def total_compensation(self):
        base = self.base_salary * self.workload_percent
        return base * (1 + (self.bonus_percent/100.0))

class ManagerCalc(EmployeeCalc):
    def total_compensation(self):
        return super().total_compensation() * 1.05  # +5% extra

class SeniorManagerCalc(ManagerCalc):
    def total_compensation(self):
        return super().total_compensation() * 1.10  # +10% extra

print("Employee 60k +10%:", EmployeeCalc(60000,10).total_compensation())
print("Manager 60k +10% :", ManagerCalc(60000,10).total_compensation())
print("Senior 60k +10%  :", SeniorManagerCalc(60000,10).total_compensation())



### 4) `__str__` leggibile
Implementa `__str__` per stampa user‑friendly (nome completo, ruolo, reparto), lasciando `__repr__` più tecnico per debug.  
Esempio:
```py
def __str__(self):
    return f"{self.employee_id} – {self.full_name} ({self.role.name}, {self.department.name})"
```



### Checklist Parte A
- [ ] Validazioni complete in `Worker.validate()`
- [ ] Property `annual_bonus_value` su `Employee`
- [ ] `Manager` override (+5%); **opzionale** `SeniorManager` (+10%)
- [ ] `__str__` leggibile senza rompere `__repr__`



# Parte B — Pipeline Pandas & Report (`hr_analysis.py`)

### 1) `load_employees_csv(path)`
- Tipizza le colonne (int/float/datetime).
- Verifica che `employee_id` sia **unico**.
- Applica **valori ammessi** (es. `contract_type` in `{"employee","manager","contractor"}`; `currency` ammessa; `workload_percent` in (0,1]).
- In caso di problemi → alza `SchemaValidationError` (o eccezioni di dominio coerenti).


In [ ]:

# ESEMPIO: schema + valori ammessi con pandas
import pandas as pd
import numpy as np
from datetime import datetime

data = [
    dict(employee_id=1, role="Developer", level=2, department="Engineering",
         cost_center="CC100", contract_type="employee", base_salary=60000,
         bonus_percent=10.0, workload_percent=1.0, currency="EUR", gender="F",
         first_name="Alice", last_name="Rossi", hire_date="2021-01-15"),
    dict(employee_id=2, role="Manager", level=3, department="Sales",
         cost_center="CC200", contract_type="manager", base_salary=70000,
         bonus_percent=12.0, workload_percent=1.0, currency="EUR", gender="M",
         first_name="Franco", last_name="Marroni", hire_date="2019-05-10"),
]
df_ex = pd.DataFrame(data)
df_ex["hire_date"] = pd.to_datetime(df_ex["hire_date"])

ALLOWED_CONTRACT = {"employee","manager","contractor"}
ALLOWED_CURRENCY = {"EUR","USD","RON"}

# Unicità
assert df_ex["employee_id"].is_unique, "employee_id duplicati"

# Valori ammessi
assert set(df_ex["contract_type"]).issubset(ALLOWED_CONTRACT), "contract_type non valido"
assert set(df_ex["currency"]).issubset(ALLOWED_CURRENCY), "currency non valida"

# Tipi numerici coerenti
df_ex["level"] = df_ex["level"].astype(int)
df_ex["base_salary"] = df_ex["base_salary"].astype(float)
df_ex["bonus_percent"] = df_ex["bonus_percent"].astype(float)
df_ex["workload_percent"] = df_ex["workload_percent"].astype(float)

df_ex.head()



### 2) `compute_total_compensation(df)` con seniority e clip bonus
Estensioni suggerite:
- `seniority_years` = anni da `hire_date` (approssimati) → extra **+0.5% per anno**, max **+5%**.
- **Cap** del bonus totale (es. tra 0% e 30%).
- `contractor` → ignora i bonus.

> Implementa in modo **additivo** (non rompere i test esistenti).


In [ ]:

# ESEMPIO: seniority, clip bonus e regole per contractor
import numpy as np
import pandas as pd

df = df_ex.copy()

today = pd.Timestamp("2025-01-01")
df["seniority_years"] = ((today - df["hire_date"]).dt.days / 365.25).clip(lower=0).round(0)
df["extra_from_seniority"] = (0.5 * df["seniority_years"]).clip(upper=5.0)  # in percento

# bonus percent finale (clippato 0..30)
df["bonus_total_percent"] = (df["bonus_percent"] + df["extra_from_seniority"]).clip(0, 30)

def total_comp_row(r):
    base = r["base_salary"] * r["workload_percent"]
    if r["contract_type"] == "contractor":
        return base
    return base * (1 + r["bonus_total_percent"]/100.0)

df["total_compensation"] = df.apply(total_comp_row, axis=1)
df[["first_name","role","seniority_years","bonus_total_percent","total_compensation"]]



### 3) `department_stats(df)` con nuove metriche e ranking
Aggiungi:
- `std` (deviazione standard)
- `iqr` (InterQuartile Range = `q75 - q25`)
- `rank` per media **discendente**; ordina il DataFrame per `mean` desc.


In [ ]:

# ESEMPIO: groupby con std, IQR e ranking
g = df.groupby("department")["total_compensation"]
stats = pd.DataFrame({
    "count": g.count(),
    "mean": g.mean().round(2),
    "median": g.median().round(2),
    "min": g.min().round(2),
    "max": g.max().round(2),
    "std": g.std(ddof=1).round(2),
})
q75 = g.quantile(0.75)
q25 = g.quantile(0.25)
stats["iqr"] = (q75 - q25).round(2)

stats = stats.reset_index().sort_values("mean", ascending=False)
stats["rank"] = stats["mean"].rank(method="dense", ascending=False).astype(int)
stats



### 4) `generate_reports(...)`: CSV + Markdown + **JSON** + **Grafico**
- Esporta anche `department_salary_summary.json`
- Crea un grafico a barre `department_salary_summary.png` con la **media per reparto** (usa `matplotlib`).


In [ ]:

# ESEMPIO: export CSV/JSON e grafico (dimostrativo, usa la variabile `stats` calcolata sopra)
import json
import os
import matplotlib.pyplot as plt

OUT_DIR = "/mnt/data/hr_demo_reports"
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, "department_salary_summary.csv")
json_path = os.path.join(OUT_DIR, "department_salary_summary.json")
png_path = os.path.join(OUT_DIR, "department_salary_summary.png")

stats.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(stats.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# Grafico: un'unica figura, nessun colore specificato
plt.figure()
plt.bar(stats["department"], stats["mean"])
plt.title("Retribuzione media per reparto")
plt.xlabel("Reparto")
plt.ylabel("Media (EUR)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(png_path)
png_path, csv_path, json_path



## Suggerimenti finali
- Mantieni le **estensioni additive**: non modificare le firme dei metodi usati nei test.
- Quando aggiungi colonne derivate (es. `seniority_years`), evita di cambiare quelle già usate dai test.
- Per i bonus, usa `np.clip`/`Series.clip` per applicare limiti puliti.
- I report extra (JSON, PNG) non devono rompere quelli già attesi (CSV, MD).
